# Master instruction —Feature Engineering-breed

Je werkt als senior research assistant voor een masterthesis in data analysis.
Voor meer informatie over de thesis / onderzoeksvoorstel / opzet: bekijk bron "Geannoteerd_onderzoeksvoorstel.md" en voor extra gelinkte literatuur bron; "External factors and SHAP in Urban Parking copy.pdf" (beide bestanden zijn te vinden in de map 'literatuur_en_info' (binnen dit project)) 

voor structuur en gewenste flow check projectalomvattede: "README.md"

Context:
- Check hieronder de vooropgestelde feature engineering notebookstructuur:
  1. fe_00_design_and_split_lock.ipynb
  2. fe_01_build_core_features_TSE.ipynb
  3. fe_02_build_forecast_lag_features_L.ipynb
  4. fe_03_finalize_feature_sets_and_export.ipynb
- Projectfase: Phase 3 — Feature Engineering
- Phase 2 (EDA) is afgerond, dit is DE basis voor interne consistentie en keuzes voor FE!!
  - Belangrijk is ook gelijkaardige stijl als phase 2 in phase 3 over te nemen, maar dan specifiek op feature engineering toegepast natuurlijk
- shortterm is de primaire modelbasis
  - (longterm blijft enkel robustness-context en mag niet het hoofddesign sturen)
- Er zijn twee modeltracks (meer uitleg is te vinden in: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/literatuur_en_info/Geannoteerd_onderzoeksvoorstel copy.md):
  1. forecast track: autoregressieve features toegestaan
  2. policy track: occupancy-lags NIET toegestaan
- De operationele split ligt vast:
  - train = 2020, 2023, 2024
  - holdout = 2025
  - 2019 uitgesloten wegens volledig missende weatherfeatures
- Alle transformaties moeten train-only gefit worden
- Alle tijdsfeatures moeten strikt causaal opgebouwd worden
- Model- en featurekeuze gebeurt later via rolling/blocked CV binnen train
- 2025 blijft volledig locked voor finale evaluatie

Doel van phase 3:
- een reproduceerbare, leakage-safe, immutable featureset bouwen
- consistente schema’s afleveren voor forecast en policy
- features expliciet structureren in blokken (cfr. onderzoeksvoorstel):
  - T = time structure
  - S = spatial identity
  - E = external factors
  - L = autoregressive structure (alleen forecast)
- alle featurekeuzes moeten inhoudelijk verdedigbaar zijn vanuit EDA en onderzoeksdoel

Belangrijke werkinstructies:
1. Werk notebook-native en reproduceerbaar.
2. Gebruik enkel train-only fitting voor alle fit-afhankelijke transformaties.
3. Splits train en holdout vroeg en hard; nooit lekken van 2025 naar train.
4. Maak feature engineering expliciet causaal:
   - geen toekomstinformatie
   - geen target-proxies
   - geen smoothing over toekomstige observaties
   - geen niet-causale aggregaties
5. Gebruik duidelijke helperfuncties en schema-validatie.
6. Werk primair op shortterm.
7. longterm mag enkel gebruikt worden voor beperkte robustness-context, niet als primaire pipelinebasis.
8. Maak alle featurebeslissingen expliciet en academisch verdedigbaar.
9. Sluit elk notebook af met:
   - "What was built"
   - "Leakage and validity checks"
   - "Implications for next notebook"
10. Indien iets ambigu is, kies de meest leakage-veilige optie en documenteer die.

Schrijf elke interpretatieve markdown-sectie alsof ze later kan worden herwerkt tot tekst voor de masterthesis.

Stijlregels:
- helder, academisch, voorzichtig
- geen losse bullet dump als lopende tekst beter is
- benoem richting, grootteorde, onzekerheid en beperking
- maak expliciet waarom het resultaat relevant is voor de volgende fase

Jouw taak is niet alleen code schrijven, maar een thesis-waardige feature engineering pipeline ontwerpen.

## Notebookspecifieke prompt
Maak notebook `fe_02_build_forecast_lag_features_L.ipynb`.

Doel:
Alleen de forecast-only autoregressieve features bouwen, strikt causaal en leakage-safe.

Context:
- Dit notebook werkt bovenop de reeds gebouwde TSE-featurelaag
- Deze features zijn uitsluitend toegestaan in de forecast track
- Policy track mag deze features nooit krijgen

Voer dit uit:
0. Analyseer eersrt al eens wat je hebt gedaan in fe_01 en fe_00 zodat je geen overbodige dingen doet, of inconsistente , incongruente keuzes / beslsisingen, kritieke fouten vermijden uiteraard!!
1. Laad de TSE-train en TSE-holdout featurelagen.
2. Bouw strikt causale occupancy-lags:
   - occ_lag_1h
   - occ_lag_24h
   - occ_lag_168h
3. Bouw eventueel bijkomende autoregressieve features enkel indien strikt verdedigbaar:
   - causal rolling mean based only on past values
   - causal rolling std
   - lag deltas
   - same-hour previous-day / previous-week variants
4. Zorg dat alle lagberekeningen:
   - per parking gebeuren
   - time-aware zijn
   - gaten correct respecteren
   - geen forward fill of verborgen leakage bevatten
5. Definieer geldigheidsregels:
   - wanneer is een lag geldig?
   - wanneer moet NaN blijven?
   - wanneer mag een observatie in forecast modelling later uitgesloten worden?
6. Onderzoek of 168h-lag filtering de sample sterk verandert en documenteer dit.
7. Maak een expliciet forecast-only registry van alle L-features.
8. Maak een formele scheiding:
   - policy_features = TSE
   - forecast_features = TSE + L
9. Exporteer forecast train/holdout features apart.
10. Sluit af met:
   - "Autoregressive feature decisions"
   - "Sample loss and validity implications"
   - "Why these features are forecast-only"

Belangrijk:
- Geen enkele lag of rolling mag impliciet information leakage introduceren.
- Als rolling features twijfelachtig zijn, kies conservatief.
- Voeg alleen L-features toe die waarschijnlijk echte meerwaarde hebben boven complexiteit.

## 0. Setup and strict scope

This notebook builds **forecast-only** autoregressive features (`L`) on top of the immutable `TSE` layer from `fe_01`.

Non-negotiable constraints:
- no policy access to `L` features;
- strictly causal construction;
- no forward fill;
- time-aware lagging per `parking_id` with exact timestamp alignment.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found")


def ensure_datetime(df: pd.DataFrame, col: str) -> None:
    df[col] = pd.to_datetime(df[col], errors="coerce")


def add_time_aware_lag(
    df: pd.DataFrame,
    lag_hours: int,
    value_col: str = "occupancy_rate",
    id_col: str = "parking_id",
    time_col: str = "rounded_hour",
) -> pd.DataFrame:
    lag_col = f"l_occ_lag_{lag_hours}h"
    lookup = df[[id_col, time_col, value_col]].rename(columns={value_col: lag_col}).copy()
    lookup[time_col] = lookup[time_col] + pd.Timedelta(hours=lag_hours)
    return df.merge(lookup, on=[id_col, time_col], how="left")


def add_strict_contiguous_rolling_features(
    df: pd.DataFrame,
    id_col: str = "parking_id",
    time_col: str = "rounded_hour",
    value_col: str = "occupancy_rate",
) -> pd.DataFrame:
    out = df.sort_values([id_col, time_col]).copy()

    prev_time = out.groupby(id_col)[time_col].shift(1)
    is_step_1h = prev_time.notna() & ((out[time_col] - prev_time) == pd.Timedelta(hours=1))
    out["_contig_break"] = ~is_step_1h
    out["_contig_id"] = out.groupby(id_col)["_contig_break"].cumsum()

    out["l_occ_rollmean_24h_strict"] = (
        out.groupby([id_col, "_contig_id"])[value_col]
        .transform(lambda s: s.shift(1).rolling(window=24, min_periods=24).mean())
    )

    out["l_occ_rollstd_24h_strict"] = (
        out.groupby([id_col, "_contig_id"])[value_col]
        .transform(lambda s: s.shift(1).rolling(window=24, min_periods=24).std(ddof=0))
    )

    out = out.drop(columns=["_contig_break", "_contig_id"])
    return out


def lag_validity_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["l_valid_lag_1h"] = out["l_occ_lag_1h"].notna().astype(np.int8)
    out["l_valid_lag_24h"] = out["l_occ_lag_24h"].notna().astype(np.int8)
    out["l_valid_lag_168h"] = out["l_occ_lag_168h"].notna().astype(np.int8)

    out["l_valid_all_core"] = (
        (out["l_valid_lag_1h"] == 1)
        & (out["l_valid_lag_24h"] == 1)
        & (out["l_valid_lag_168h"] == 1)
    ).astype(np.int8)

    out["l_valid_roll24_strict"] = (
        out["l_occ_rollmean_24h_strict"].notna() & out["l_occ_rollstd_24h_strict"].notna()
    ).astype(np.int8)

    out["l_valid_core_plus_roll24"] = (
        (out["l_valid_all_core"] == 1) & (out["l_valid_roll24_strict"] == 1)
    ).astype(np.int8)

    return out


PROJECT_ROOT = find_project_root()
TSE_DIR = PROJECT_ROOT / "data_processed" / "phase3_features" / "core_tse"
SPLIT_DIR = PROJECT_ROOT / "data_processed" / "phase3_splits"
OUT_DIR = PROJECT_ROOT / "data_processed" / "phase3_features" / "forecast_l"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("TSE dir:", TSE_DIR)
print("Output dir:", OUT_DIR)

Project root: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2
TSE dir: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/phase3_features/core_tse
Output dir: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/phase3_features/forecast_l


## 1. Consistency check with `fe_00` and `fe_01`

Before building `L`, we verify existing contracts so no inconsistent decisions are introduced.

In [2]:
split_summary_path = SPLIT_DIR / "shortterm_operational_split_summary.csv"
tse_manifest_path = TSE_DIR / "tse_export_manifest.csv"
tse_registry_path = TSE_DIR / "feature_registry_tse.csv"

split_summary_df = pd.read_csv(split_summary_path)
tse_manifest_df = pd.read_csv(tse_manifest_path)
tse_registry_df = pd.read_csv(tse_registry_path)

contract_check_df = pd.DataFrame(
    [
        {
            "contract": "Locked split from fe_00",
            "expected": "train years 2020/2023/2024 and holdout year 2025",
            "status": "PASS" if set(split_summary_df["operational_split"]) == {"train", "holdout"} else "CHECK",
            "evidence": f"rows={len(split_summary_df)}",
        },
        {
            "contract": "TSE exists from fe_01",
            "expected": "train and holdout TSE parquet exported",
            "status": "PASS" if {"train_tse_parquet", "holdout_tse_parquet"}.issubset(set(tse_manifest_df["artifact"])) else "CHECK",
            "evidence": f"artifacts={len(tse_manifest_df)}",
        },
        {
            "contract": "No L-feature already in TSE",
            "expected": "no lag/rolling columns in fe_01 export",
            "status": "PASS" if not tse_registry_df["feature_name"].str.contains("lag|roll", case=False, regex=True).any() else "CHECK",
            "evidence": "registry scan on feature_name",
        },
    ]
)

print("Contract checks")
display(contract_check_df)

print("fe_00 split summary")
display(split_summary_df)

print("fe_01 TSE manifest")
display(tse_manifest_df)

Contract checks


,contract,expected,status,evidence
0,Locked split from fe_00,train years 2020/2023/2024 and holdout year 2025,PASS,rows=2
1,TSE exists from fe_01,train and holdout TSE parquet exported,PASS,artifacts=6
2,No L-feature already in TSE,no lag/rolling columns in fe_01 export,PASS,registry scan on feature_name


fe_00 split summary


,operational_split,n_rows,n_parkings,date_min,date_max,mean_occupancy,pct_of_phase3_scope
0,holdout,87600,10,2025-01-01,2025-12-31 23:00:00,0.400808,40.287532
1,train,129837,10,2020-01-01,2024-12-31 23:00:00,0.361632,59.712468


fe_01 TSE manifest


,artifact,path,n_rows,n_cols
0,train_tse_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,129837,92
1,holdout_tse_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,87600,92
2,feature_registry_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,87,9
3,data_dictionary_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,87,6
4,fit_params_json,/Users/emilevandevoorde/Documents/mp_mechelen_...,1,1
5,schema_json,/Users/emilevandevoorde/Documents/mp_mechelen_...,1,1


## 2. Load TSE train and holdout layers

`policy_features = TSE` remains unchanged.
`forecast_features` will be built as `TSE + L` in this notebook.

In [3]:
train_tse_path = TSE_DIR / "tse_core_train_2020_2023_2024.parquet"
holdout_tse_path = TSE_DIR / "tse_core_holdout_2025.parquet"

train_tse = pd.read_parquet(train_tse_path)
holdout_tse = pd.read_parquet(holdout_tse_path)

for df in [train_tse, holdout_tse]:
    ensure_datetime(df, "rounded_hour")
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

scope_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_tse),
            "n_parkings": train_tse["parking_id"].nunique(),
            "years": ", ".join(map(str, sorted(train_tse["year"].dropna().astype(int).unique().tolist()))),
            "date_min": train_tse["rounded_hour"].min(),
            "date_max": train_tse["rounded_hour"].max(),
        },
        {
            "split": "holdout",
            "rows": len(holdout_tse),
            "n_parkings": holdout_tse["parking_id"].nunique(),
            "years": ", ".join(map(str, sorted(holdout_tse["year"].dropna().astype(int).unique().tolist()))),
            "date_min": holdout_tse["rounded_hour"].min(),
            "date_max": holdout_tse["rounded_hour"].max(),
        },
    ]
)

display(scope_df)

,split,rows,n_parkings,years,date_min,date_max
0,train,129837,10,"2020, 2023, 2024",2020-01-01,2024-12-31 23:00:00
1,holdout,87600,10,2025,2025-01-01,2025-12-31 23:00:00


## 3. Build strictly causal occupancy lags

Core lags:
- `l_occ_lag_1h`
- `l_occ_lag_24h`
- `l_occ_lag_168h`

Implementation choice:
- lags are built on the combined train+holdout timeline to allow valid holdout boundary history (still strictly past-only).
- no forward fill; exact timestamp match is required.

In [4]:
combined_tse = pd.concat([train_tse, holdout_tse], axis=0, ignore_index=True)
combined_tse = combined_tse.sort_values(["parking_id", "rounded_hour"]).reset_index(drop=True)

for lag_h in [1, 24, 168]:
    combined_tse = add_time_aware_lag(combined_tse, lag_hours=lag_h)

# additional conservative AR candidates
combined_tse = add_strict_contiguous_rolling_features(combined_tse)
combined_tse["l_occ_lag_delta_1h_24h"] = combined_tse["l_occ_lag_1h"] - combined_tse["l_occ_lag_24h"]
combined_tse["l_occ_lag_delta_24h_168h"] = combined_tse["l_occ_lag_24h"] - combined_tse["l_occ_lag_168h"]

combined_tse = lag_validity_flags(combined_tse)

train_l = combined_tse.loc[combined_tse["operational_split"].eq("train")].copy()
holdout_l = combined_tse.loc[combined_tse["operational_split"].eq("holdout")].copy()

lag_missing_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "missing_lag_1h_pct": float(train_l["l_occ_lag_1h"].isna().mean() * 100),
            "missing_lag_24h_pct": float(train_l["l_occ_lag_24h"].isna().mean() * 100),
            "missing_lag_168h_pct": float(train_l["l_occ_lag_168h"].isna().mean() * 100),
            "missing_rollmean_24h_strict_pct": float(train_l["l_occ_rollmean_24h_strict"].isna().mean() * 100),
            "missing_rollstd_24h_strict_pct": float(train_l["l_occ_rollstd_24h_strict"].isna().mean() * 100),
        },
        {
            "split": "holdout",
            "missing_lag_1h_pct": float(holdout_l["l_occ_lag_1h"].isna().mean() * 100),
            "missing_lag_24h_pct": float(holdout_l["l_occ_lag_24h"].isna().mean() * 100),
            "missing_lag_168h_pct": float(holdout_l["l_occ_lag_168h"].isna().mean() * 100),
            "missing_rollmean_24h_strict_pct": float(holdout_l["l_occ_rollmean_24h_strict"].isna().mean() * 100),
            "missing_rollstd_24h_strict_pct": float(holdout_l["l_occ_rollstd_24h_strict"].isna().mean() * 100),
        },
    ]
)

print("Lag/rolling missingness summary")
display(lag_missing_summary_df.round(3))

Lag/rolling missingness summary


,split,missing_lag_1h_pct,missing_lag_24h_pct,missing_lag_168h_pct,missing_rollmean_24h_strict_pct,missing_rollstd_24h_strict_pct
0,train,5.772,7.409,9.158,54.882,54.882
1,holdout,0.000,0.000,0.000,0.000,0.000


## 4. Time-aware and leakage integrity checks

Checks performed:
- lags are computed per parking;
- no duplicate key rows;
- no forward-filled lag values;
- lag validity requires exact timestamp availability at `t-h`.

In [5]:
key_dupes_train = int(train_l.duplicated(["parking_id", "rounded_hour"]).sum())
key_dupes_holdout = int(holdout_l.duplicated(["parking_id", "rounded_hour"]).sum())

# explicit alignment check for lag 1h and 168h
base_lookup = combined_tse[["parking_id", "rounded_hour", "occupancy_rate"]].copy()

def validate_lag_alignment(df: pd.DataFrame, lag_hours: int, lag_col: str) -> float:
    chk = df[["parking_id", "rounded_hour", lag_col]].copy()
    chk = chk.loc[chk[lag_col].notna()].copy()
    if chk.empty:
        return 0.0

    lookup = base_lookup.rename(columns={"occupancy_rate": "expected_lag_value"}).copy()
    lookup["rounded_hour"] = lookup["rounded_hour"] + pd.Timedelta(hours=lag_hours)

    chk = chk.merge(lookup, on=["parking_id", "rounded_hour"], how="left")
    mismatch = (~np.isclose(chk[lag_col].astype(float), chk["expected_lag_value"].astype(float), equal_nan=True)).mean()
    return float(mismatch)

mismatch_1h = validate_lag_alignment(combined_tse, 1, "l_occ_lag_1h")
mismatch_168h = validate_lag_alignment(combined_tse, 168, "l_occ_lag_168h")

integrity_df = pd.DataFrame(
    [
        {"check": "duplicate key rows in train", "result": "PASS" if key_dupes_train == 0 else "FAIL", "detail": key_dupes_train},
        {"check": "duplicate key rows in holdout", "result": "PASS" if key_dupes_holdout == 0 else "FAIL", "detail": key_dupes_holdout},
        {"check": "lag 1h value-source alignment", "result": "PASS" if mismatch_1h == 0 else "FAIL", "detail": mismatch_1h},
        {"check": "lag 168h value-source alignment", "result": "PASS" if mismatch_168h == 0 else "FAIL", "detail": mismatch_168h},
    ]
)

display(integrity_df)

,check,result,detail
0,duplicate key rows in train,PASS,0.0
1,duplicate key rows in holdout,PASS,0.0
2,lag 1h value-source alignment,PASS,0.0
3,lag 168h value-source alignment,PASS,0.0


## 5. Validity rules for lag usage

Formal rules used later in forecast modelling:
- `l_occ_lag_1h` valid only if same parking has observation at `t-1h`.
- `l_occ_lag_24h` valid only if same parking has observation at `t-24h`.
- `l_occ_lag_168h` valid only if same parking has observation at `t-168h`.
- no synthetic imputation for missing lags in this notebook.
- strict-core training subset can be defined by `l_valid_all_core == 1`.

In [6]:
validity_rules_df = pd.DataFrame(
    [
        {
            "rule_id": "VR1",
            "condition": "l_occ_lag_1h not null",
            "meaning": "1-hour exact historical occupancy exists for same parking",
            "action_if_false": "keep as NaN; optional exclusion in model training",
        },
        {
            "rule_id": "VR2",
            "condition": "l_occ_lag_24h not null",
            "meaning": "same-hour previous day occupancy exists",
            "action_if_false": "keep as NaN; optional exclusion in model training",
        },
        {
            "rule_id": "VR3",
            "condition": "l_occ_lag_168h not null",
            "meaning": "same-hour previous week occupancy exists",
            "action_if_false": "keep as NaN; required for strict weekly-lag filtering",
        },
        {
            "rule_id": "VR4",
            "condition": "l_valid_all_core == 1",
            "meaning": "all three core lags available",
            "action_if_false": "row may be excluded in strict forecast training protocol",
        },
        {
            "rule_id": "VR5",
            "condition": "l_valid_core_plus_roll24 == 1",
            "meaning": "core lags + strict 24h rolling window available",
            "action_if_false": "rolling-based model variant should exclude row",
        },
    ]
)

display(validity_rules_df)

,rule_id,condition,meaning,action_if_false
0,VR1,l_occ_lag_1h not null,1-hour exact historical occupancy exists for s...,keep as NaN; optional exclusion in model training
1,VR2,l_occ_lag_24h not null,same-hour previous day occupancy exists,keep as NaN; optional exclusion in model training
2,VR3,l_occ_lag_168h not null,same-hour previous week occupancy exists,keep as NaN; required for strict weekly-lag fi...
3,VR4,l_valid_all_core == 1,all three core lags available,row may be excluded in strict forecast trainin...
4,VR5,l_valid_core_plus_roll24 == 1,core lags + strict 24h rolling window available,rolling-based model variant should exclude row


## 6. 168h-lag filtering impact analysis

We quantify sample impact of `l_occ_lag_168h` validity and compare two holdout constructions:
1. causal combined-history lagging (chosen);
2. holdout-only lagging (for boundary comparison).

In [7]:
def sample_loss_table(df: pd.DataFrame, split_name: str) -> dict[str, object]:
    n_total = int(len(df))
    n_valid_168 = int((df["l_valid_lag_168h"] == 1).sum())
    n_drop_168 = n_total - n_valid_168
    return {
        "split": split_name,
        "n_total": n_total,
        "n_valid_168": n_valid_168,
        "n_drop_168": n_drop_168,
        "pct_drop_168": (n_drop_168 / n_total * 100) if n_total > 0 else 0.0,
    }

sample_loss_df = pd.DataFrame(
    [
        sample_loss_table(train_l, "train"),
        sample_loss_table(holdout_l, "holdout"),
    ]
)

impact_by_year_df = (
    combined_tse.groupby(["operational_split", "year"], as_index=False)["l_valid_lag_168h"]
    .mean()
    .rename(columns={"l_valid_lag_168h": "valid_rate_168"})
)

impact_by_parking_df = (
    combined_tse.groupby(["operational_split", "parking_id"], as_index=False)["l_valid_lag_168h"]
    .mean()
    .rename(columns={"l_valid_lag_168h": "valid_rate_168"})
    .sort_values(["operational_split", "valid_rate_168", "parking_id"])
)

# Boundary comparison: combined-history vs holdout-only lag 168
holdout_only = holdout_tse.sort_values(["parking_id", "rounded_hour"]).copy()
lookup_168 = holdout_only[["parking_id", "rounded_hour", "occupancy_rate"]].rename(columns={"occupancy_rate": "holdout_only_lag_168"})
lookup_168["rounded_hour"] = lookup_168["rounded_hour"] + pd.Timedelta(hours=168)
holdout_only = holdout_only.merge(lookup_168, on=["parking_id", "rounded_hour"], how="left")

holdout_compare = holdout_l[["parking_id", "rounded_hour", "l_occ_lag_168h"]].merge(
    holdout_only[["parking_id", "rounded_hour", "holdout_only_lag_168"]],
    on=["parking_id", "rounded_hour"],
    how="left",
)

comparison_df = pd.DataFrame(
    [
        {
            "metric": "valid_rate_168_combined_history",
            "value": float(holdout_compare["l_occ_lag_168h"].notna().mean()),
        },
        {
            "metric": "valid_rate_168_holdout_only",
            "value": float(holdout_compare["holdout_only_lag_168"].notna().mean()),
        },
        {
            "metric": "rows_recovered_by_combined_history",
            "value": int((holdout_compare["l_occ_lag_168h"].notna() & holdout_compare["holdout_only_lag_168"].isna()).sum()),
        },
    ]
)

print("Sample loss due to 168h validity")
display(sample_loss_df.round(4))

print("168h validity by year")
display(impact_by_year_df.round(4))

print("168h validity by parking")
display(impact_by_parking_df.round(4))

print("Holdout boundary comparison")
display(comparison_df)

Sample loss due to 168h validity


,split,n_total,n_valid_168,n_drop_168,pct_drop_168
0,train,129837,117947,11890,9.1576
1,holdout,87600,87600,0,0.0000


168h validity by year


,operational_split,year,valid_rate_168
0,holdout,2025,1.0000
1,train,2020,0.8858
2,train,2023,0.8549
3,train,2024,0.9589


168h validity by parking


,operational_split,parking_id,valid_rate_168
0,holdout,P Bruul,1.0000
1,holdout,P Grote Markt,1.0000
2,holdout,P Hoogstraat,1.0000
3,holdout,P Kathedraal,1.0000
4,holdout,P Keerdok,1.0000
5,holdout,P Komet,1.0000
6,holdout,P Lamot,1.0000
7,holdout,P Maarten,1.0000
8,holdout,P Tinel,1.0000
9,holdout,P Veemarkt,1.0000


Holdout boundary comparison


,metric,value
0,valid_rate_168_combined_history,1.000000
1,valid_rate_168_holdout_only,0.980822
2,rows_recovered_by_combined_history,1680.000000


## 7. Forecast-only L-feature registry

We keep a conservative `L` set:
- selected for export: core lags + lag deltas;
- computed but **not selected by default**: strict rolling features (high sample impact);
- not added as separate columns: same-hour aliases (duplicate information vs lag 24/168).

In [8]:
l_registry_rows = [
    {
        "feature_name": "l_occ_lag_1h",
        "block": "L",
        "source_columns": "occupancy_rate@t-1h (same parking, exact timestamp)",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "very-short memory autoregression",
        "notes": "selected_for_export",
    },
    {
        "feature_name": "l_occ_lag_24h",
        "block": "L",
        "source_columns": "occupancy_rate@t-24h (same parking, exact timestamp)",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "daily seasonality memory",
        "notes": "selected_for_export",
    },
    {
        "feature_name": "l_occ_lag_168h",
        "block": "L",
        "source_columns": "occupancy_rate@t-168h (same parking, exact timestamp)",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "weekly seasonality memory",
        "notes": "selected_for_export",
    },
    {
        "feature_name": "l_occ_lag_delta_1h_24h",
        "block": "L",
        "source_columns": "l_occ_lag_1h,l_occ_lag_24h",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "short-vs-daily regime contrast",
        "notes": "selected_for_export",
    },
    {
        "feature_name": "l_occ_lag_delta_24h_168h",
        "block": "L",
        "source_columns": "l_occ_lag_24h,l_occ_lag_168h",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "daily-vs-weekly regime contrast",
        "notes": "selected_for_export",
    },
    {
        "feature_name": "l_occ_rollmean_24h_strict",
        "block": "L",
        "source_columns": "past 24 contiguous hourly occupancy values",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "local level smoothing",
        "notes": "computed_not_selected_default_due_sample_loss",
    },
    {
        "feature_name": "l_occ_rollstd_24h_strict",
        "block": "L",
        "source_columns": "past 24 contiguous hourly occupancy values",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "local volatility",
        "notes": "computed_not_selected_default_due_sample_loss",
    },
]

l_registry_df = pd.DataFrame(l_registry_rows)

selected_l_features = l_registry_df.loc[
    l_registry_df["notes"].str.contains("selected_for_export"), "feature_name"
].tolist()

computed_but_not_selected = l_registry_df.loc[
    l_registry_df["notes"].str.contains("not_selected"), "feature_name"
].tolist()

print("L-feature registry")
display(l_registry_df)

print("Selected L features:", selected_l_features)
print("Computed but not selected by default:", computed_but_not_selected)

L-feature registry


,feature_name,block,source_columns,track_allowed,requires_fit,train_only_fit,causal,expected_role,notes
0,l_occ_lag_1h,L,"occupancy_rate@t-1h (same parking, exact times...",forecast,False,False,True,very-short memory autoregression,selected_for_export
1,l_occ_lag_24h,L,"occupancy_rate@t-24h (same parking, exact time...",forecast,False,False,True,daily seasonality memory,selected_for_export
2,l_occ_lag_168h,L,"occupancy_rate@t-168h (same parking, exact tim...",forecast,False,False,True,weekly seasonality memory,selected_for_export
3,l_occ_lag_delta_1h_24h,L,"l_occ_lag_1h,l_occ_lag_24h",forecast,False,False,True,short-vs-daily regime contrast,selected_for_export
4,l_occ_lag_delta_24h_168h,L,"l_occ_lag_24h,l_occ_lag_168h",forecast,False,False,True,daily-vs-weekly regime contrast,selected_for_export
5,l_occ_rollmean_24h_strict,L,past 24 contiguous hourly occupancy values,forecast,False,False,True,local level smoothing,computed_not_selected_default_due_sample_loss
6,l_occ_rollstd_24h_strict,L,past 24 contiguous hourly occupancy values,forecast,False,False,True,local volatility,computed_not_selected_default_due_sample_loss


Selected L features: ['l_occ_lag_1h', 'l_occ_lag_24h', 'l_occ_lag_168h', 'l_occ_lag_delta_1h_24h', 'l_occ_lag_delta_24h_168h']
Computed but not selected by default: ['l_occ_rollmean_24h_strict', 'l_occ_rollstd_24h_strict']


## 8. Formal feature separation

- `policy_features = TSE`
- `forecast_features = TSE + selected L`

In [9]:
policy_feature_cols = [
    c for c in train_tse.columns if c.startswith("t_") or c.startswith("s_") or c.startswith("e_")
]

forecast_feature_cols = policy_feature_cols + selected_l_features

separation_df = pd.DataFrame(
    [
        {
            "set_name": "policy_features",
            "n_features": len(policy_feature_cols),
            "includes_l_features": any(c.startswith("l_") for c in policy_feature_cols),
            "definition": "TSE only",
        },
        {
            "set_name": "forecast_features",
            "n_features": len(forecast_feature_cols),
            "includes_l_features": any(c.startswith("l_") for c in forecast_feature_cols),
            "definition": "TSE + selected L",
        },
    ]
)

assert not any(c.startswith("l_") for c in policy_feature_cols), "Policy feature set contains L feature"
assert all(c.startswith(("t_", "s_", "e_", "l_")) for c in forecast_feature_cols), "Unexpected feature prefix"

print("Formal separation summary")
display(separation_df)

Formal separation summary


,set_name,n_features,includes_l_features,definition
0,policy_features,87,False,TSE only
1,forecast_features,92,True,TSE + selected L


## 9. Export forecast train/holdout features

In [10]:
# Keep selected L features plus validity flags as audit columns
validity_cols = [
    "l_valid_lag_1h",
    "l_valid_lag_24h",
    "l_valid_lag_168h",
    "l_valid_all_core",
    "l_valid_roll24_strict",
    "l_valid_core_plus_roll24",
]

core_key_cols = ["parking_id", "rounded_hour", "year", "occupancy_rate", "operational_split"]

forecast_export_cols = core_key_cols + policy_feature_cols + selected_l_features + validity_cols

train_forecast = train_l[forecast_export_cols].copy()
holdout_forecast = holdout_l[forecast_export_cols].copy()

# strict subset summary (not used as primary export, but documented)
strict_train = train_forecast.loc[train_forecast["l_valid_all_core"].eq(1)].copy()
strict_holdout = holdout_forecast.loc[holdout_forecast["l_valid_all_core"].eq(1)].copy()

train_out_path = OUT_DIR / "forecast_tsel_train_2020_2023_2024.parquet"
holdout_out_path = OUT_DIR / "forecast_tsel_holdout_2025.parquet"
registry_out_path = OUT_DIR / "feature_registry_l_forecast.csv"
validity_out_path = OUT_DIR / "forecast_l_validity_summary.csv"
schema_out_path = OUT_DIR / "forecast_tsel_schema.json"
set_definition_out_path = OUT_DIR / "feature_set_separation.json"
manifest_out_path = OUT_DIR / "forecast_l_export_manifest.csv"

train_forecast.to_parquet(train_out_path, index=False)
holdout_forecast.to_parquet(holdout_out_path, index=False)
l_registry_df.to_csv(registry_out_path, index=False)

validity_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "n_rows": len(train_forecast),
            "pct_valid_all_core": float(train_forecast["l_valid_all_core"].mean() * 100),
            "pct_valid_core_plus_roll24": float(train_forecast["l_valid_core_plus_roll24"].mean() * 100),
            "strict_rows_if_filter_all_core": len(strict_train),
        },
        {
            "split": "holdout",
            "n_rows": len(holdout_forecast),
            "pct_valid_all_core": float(holdout_forecast["l_valid_all_core"].mean() * 100),
            "pct_valid_core_plus_roll24": float(holdout_forecast["l_valid_core_plus_roll24"].mean() * 100),
            "strict_rows_if_filter_all_core": len(strict_holdout),
        },
    ]
)
validity_summary_df.to_csv(validity_out_path, index=False)

schema_payload = {
    "columns": train_forecast.columns.tolist(),
    "dtypes": {col: str(dtype) for col, dtype in train_forecast.dtypes.items()},
}
schema_out_path.write_text(json.dumps(schema_payload, indent=2))

set_definition_payload = {
    "policy_features": policy_feature_cols,
    "forecast_features": forecast_feature_cols,
    "selected_l_features": selected_l_features,
    "computed_but_not_selected_l_features": computed_but_not_selected,
}
set_definition_out_path.write_text(json.dumps(set_definition_payload, indent=2))

manifest_df = pd.DataFrame(
    [
        {"artifact": "forecast_train_parquet", "path": str(train_out_path), "n_rows": len(train_forecast), "n_cols": train_forecast.shape[1]},
        {"artifact": "forecast_holdout_parquet", "path": str(holdout_out_path), "n_rows": len(holdout_forecast), "n_cols": holdout_forecast.shape[1]},
        {"artifact": "forecast_l_registry_csv", "path": str(registry_out_path), "n_rows": len(l_registry_df), "n_cols": l_registry_df.shape[1]},
        {"artifact": "validity_summary_csv", "path": str(validity_out_path), "n_rows": len(validity_summary_df), "n_cols": validity_summary_df.shape[1]},
        {"artifact": "forecast_schema_json", "path": str(schema_out_path), "n_rows": 1, "n_cols": 1},
        {"artifact": "feature_set_separation_json", "path": str(set_definition_out_path), "n_rows": 1, "n_cols": 1},
    ]
)
manifest_df.to_csv(manifest_out_path, index=False)

print("Forecast-L export manifest")
display(manifest_df)

print("Validity summary")
display(validity_summary_df.round(4))

Forecast-L export manifest


,artifact,path,n_rows,n_cols
0,forecast_train_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,129837,103
1,forecast_holdout_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,87600,103
2,forecast_l_registry_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,7,9
3,validity_summary_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,2,5
4,forecast_schema_json,/Users/emilevandevoorde/Documents/mp_mechelen_...,1,1
5,feature_set_separation_json,/Users/emilevandevoorde/Documents/mp_mechelen_...,1,1


Validity summary


,split,n_rows,pct_valid_all_core,pct_valid_core_plus_roll24,strict_rows_if_filter_all_core
0,train,129837,83.581,43.9151,108519
1,holdout,87600,100.000,100.0000,87600


## Autoregressive feature decisions

1. Core lags (`1h`, `24h`, `168h`) are retained as the primary forecast memory block.
2. Lag deltas are retained because they add compact regime contrast without adding new leakage paths.
3. Strict 24h rolling features are computed for diagnostics but not selected by default due high train sample impact under contiguous-window validity.
4. Same-hour previous-day/week aliases are not duplicated because they are equivalent to `lag_24h` and `lag_168h`.

## Sample loss and validity implications

1. `l_occ_lag_168h` filtering has measurable train impact but limited holdout impact with causal combined-history construction.
2. If modelling enforces `l_valid_all_core == 1`, train sample shrinks while holdout remains largely preserved.
3. Strict rolling validity is materially more restrictive and should be used only in dedicated robustness variants.

## Why these features are forecast-only

1. `L` features directly encode recent occupancy history and maximize predictive inertia capture.
2. That inertia is useful for forecasting accuracy but problematic for policy-scenario sensitivity.
3. Keeping `policy = TSE` and `forecast = TSE + L` preserves methodological separation defined in `fe_00`.

## What was built

A leakage-safe forecast-only autoregressive layer on top of immutable TSE, including core lags, conservative additional AR candidates, validity masks, and formal feature-set separation artifacts.

## Leakage and validity checks

The notebook enforces exact timestamp lag matching per parking, no forward fill, and explicit validity indicators for core and strict subsets.

## Implications for next notebook

`fe_03` can finalize policy and forecast feature packages with schema governance and modelling-ready manifests, without redefining lag logic.